# Dataset suitability audit (training-free)

Applies the same checks to every multi-stage intrusion dataset, so the results fit in one table:

1. **Structure**: campaigns, attack episodes (6 h gaps), attacking hosts, attack host pairs, internal attack pairs, duplicates.
2. **Late stage**: lateral-movement flows and the internal host pairs they connect.
3. **Identity shortcut**: average precision (AP) of a lookup on (source, destination) and on the source alone.
4. **Single-flow separability**: AP of a depth-1 and a depth-4 decision tree on one flow's native features, overall and per stage.
5. **Host-based labeling**: attacking hosts labeled through at least 80% of their active hours.

Splits: leave-one-campaign-out with 3 or more campaigns (AIT), train on the first and test on the second with 2 (CIC: Wednesday → Thursday), otherwise chronological at the median attack time (DAPT2020, Unraveled). Per-stage results use benign flows plus that stage's flows.

Code: `code/python/utils/suitability_audit.py`. No neural model is trained. Expect roughly 30–60 minutes on a standard CPU runtime, most of it downloading and auditing Unraveled.

In [ ]:
# SETUP
import os
import subprocess
import sys

from google.colab import drive
drive.mount('/content/drive')

REPO_URL = 'https://github.com/tatipar/temporalgnn-nids.git'
REPO_BRANCH = 'feat/fair-comparison-corrected'
REPO_DIR = '/content/temporalgnn-nids-audit'
if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', REPO_BRANCH, REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only'], check=True)
REPO_COMMIT = subprocess.check_output(['git', '-C', REPO_DIR, 'rev-parse', 'HEAD'], text=True).strip()
sys.path.insert(0, os.path.join(REPO_DIR, 'code/python'))
print('Code revision:', REPO_COMMIT)

In [ ]:
# CONFIGURATION
RUN = {
    'cic_original': True,    # NF-CSE-CIC-IDS2018-v3 Infiltration, labels as distributed
    'cic_corrected': True,   # same flows, Liu et al. corrected labels
    'dapt2020': True,
    'ait': True,
    'unraveled': True,       # downloads ~200 CSVs from GitLab (several GB)
}
REUSE_SAVED = True           # skip datasets whose report already exists in OUT_DIR

OUT_DIR = '/content/drive/MyDrive/nids-mitre/dataset_suitability_audit_v1'
CIC_CSVS = {
    'wed': '/content/drive/MyDrive/nids-mitre/data/cicids2018-v3/cicids2018v3_wed2802.csv',
    'thu': '/content/drive/MyDrive/nids-mitre/data/cicids2018-v3/cicids2018v3_thu0103.csv',
}
# Public data are downloaded to the runtime disk. Set CACHE_ON_DRIVE = True to keep them on Drive
# (uses Drive space, but a new session does not download them again).
CACHE_ON_DRIVE = False
DATA = '/content/drive/MyDrive/nids-mitre/audit_data' if CACHE_ON_DRIVE else '/content/audit_data'
os.makedirs(OUT_DIR, exist_ok=True)

In [ ]:
# DOWNLOADS (public sources; files already present are skipped)
import importlib
from utils import suitability_audit as sa
importlib.reload(sa)

if RUN['dapt2020']:
    sa.download_dapt2020(f'{DATA}/dapt2020')
    print('DAPT2020 ready')
if RUN['ait']:
    sa.download_ait(f'{DATA}/ait')
    print('AIT-NDS ready')
if RUN['unraveled']:
    files = sa.download_unraveled(f'{DATA}/unraveled')
    print('Unraveled ready:', len(files), 'files')
!du -sh {DATA}/* 2>/dev/null

In [ ]:
# AUDITS (one dataset at a time to bound memory)
import gc
import json
import time
from pathlib import Path

import pandas as pd

LOADERS = {
    'cic_original': lambda: sa.load_cic_nfv3(CIC_CSVS, 'original'),
    'cic_corrected': lambda: sa.load_cic_nfv3(CIC_CSVS, 'corrected'),
    'dapt2020': lambda: sa.load_dapt2020(f'{DATA}/dapt2020'),
    'ait': lambda: sa.load_ait(f'{DATA}/ait'),
    'unraveled': lambda: sa.load_unraveled(f'{DATA}/unraveled'),
}
reports = {}
for key, load in LOADERS.items():
    if not RUN[key]:
        continue
    path = Path(OUT_DIR) / f'{key}.json'
    if REUSE_SAVED and path.exists():
        reports[key] = json.loads(path.read_text())
        print(f'{key}: reusing {path}')
        continue
    started = time.time()
    flows = load()
    report = sa.audit(flows)
    report['code_revision'] = REPO_COMMIT
    sa.save(report, OUT_DIR, name=key)
    reports[key] = report
    del flows
    gc.collect()
    print(f'{key}: done in {time.time() - started:.0f} s')

In [ ]:
# SUMMARY TABLES (saved next to the JSON reports)
pd.set_option('display.max_colwidth', 80)
summary = pd.DataFrame([sa.summary_row(r) for r in reports.values()])
stages = pd.DataFrame([row for r in reports.values() for row in sa.stage_rows(r)])
summary.to_csv(f'{OUT_DIR}/summary.csv', index=False)
stages.to_csv(f'{OUT_DIR}/stages.csv', index=False)
display(summary.set_index('dataset').T)
display(stages)

In [ ]:
# DETAILS: lateral movement, dominant attack pairs and hosts labeled through most of their active hours
for key, r in reports.items():
    s, h = r['structure'], r['host_labeling']
    print(f"\n=== {r['dataset']} ===")
    print('notes:', r['notes'])
    print('lateral-movement flows:', s['lateral_movement_flows'], '| internal pairs:', s['lateral_movement_internal_pairs'])
    print('lateral-movement activities:', s['lateral_movement_activities'])
    print('duplicates:', s['duplicate_rows'], 'rows,', s['duplicate_rows_labeled'], 'of them labeled')
    for st in s['stages']:
        print(f"  {st['stage']:<40} flows={st['flows']:>8} pairs={st['pairs']:>5} "
              f"top pair={st['top_pair']} ({st['top_pair_share']}) internal pairs={st['internal_pairs']}")
    print('hosts labeled in >=80% of active hours:', h.get('flagged_hosts'))

In [ ]:
# Zip the reports for download
!cd {OUT_DIR} && zip -q -r /content/dataset_suitability_audit_v1.zip . && ls -la /content/dataset_suitability_audit_v1.zip